In [17]:
from pathlib import Path

# ========== 根路径与任务设置 ==========
ROOT = Path("./results")          # ./results/gcd/*/results.csv, ./results/openset/*/results.csv
SETTING = "openset"                   # "gcd" 或 "openset"

# 每个 setting 对应要读取的任务目录；留空 [] 或 ["*"] 表示扫描该 setting 下所有含 results.csv 的目录
TASKS_MAP = {
    "gcd":     ["alup", "deepaligned", "dpn", "geoid", "loop", "sdc", "tan"],
    "openset": ["ab", "adb", "deepunk", "doc", "dyen", "knncon", "plm_ood", "scl"],
}

# method 字段/目录名的重命名（归一化后匹配）
METHOD_RENAME = {
    # gcd
    "deepaligned": "DeepAligned",
    "geoid": "GeoID",
    "sdc": "SDC",
    "dpn": "DPN",
    "tan": "TAN",
    "loop": "LOOP",
    "alup": "ALUP",

    # openset
    "doc": "DOC",
    "deepunk": "DeepUNK",
    "adb": "ADB",
    "scl": "SCL",
    "ab": "AB",
    "knncon": "KNNCon",
    "knn_con": "KNNCon",
    "dyen": "DyEn",
    "plm_ood": "LLM_OOD",   # 只是兜底名；真正会拆 Detector
}

# Method 显示名（可选）：例如 DeepAligned 显示成 DAL（和你示例一致）
METHOD_DISPLAY = {
    "DeepAligned": "DAL",
    # 其它不需要改就不写
}

# 指标
METRICS_MAP = {
    "gcd":     ["K-ACC", "N-ACC"],
    "openset": ["K-F1", "N-F1"],
}

# 固定 LCR（你主实验就是固定这个）
LCR_KEEP = {"gcd": 0.1, "openset": 1.0}

# KCR 顺序
KCR_ORDER = [0.25, 0.50, 0.75]

# 四舍五入
ROUND_DECIMALS = 2

# 输出文件
OUTPUT_TEX = f"v5_{SETTING}_main_table.tex"

# 是否高亮 top1/top2
HIGHLIGHT_TOP2 = True

# 只显示 METHOD_ORDER_MAP 里列出的（白名单 + 顺序）
METHOD_ORDER_MAP = {
    "gcd": ["DeepAligned", "GeoID", "SDC", "DPN", "TAN", "LOOP", "ALUP"],
    "openset": ["DOC", "DeepUNK", "ADB", "SCL", "AB", "KNNCon", "DyEn",
                "OpenMax", "EnergyBased", "MaxLogit"],
}

# ========== 数据集列顺序（11个）与显示名（表头）==========
# 这里的名字用于匹配 dataset 列（大小写不敏感），显示名用于 LaTeX 表头
DATASETS_ORDER = [
    "20NG",
    "thucnews",
    "Yahoo",
    "TREC",
    "banking",
    "stackoverflow",
    "clinc",
    "hwu",
    "ecdt",
    "mcid",
    "DBPedia",
]

DATASET_DISPLAY = {
    "20NG": "20NG",
    "thucnews": "THUCNews",
    "Yahoo": "Yahoo",
    "TREC": "TREC",
    "banking": "BANK",
    "stackoverflow": "S.O.",
    "clinc": "CLINC",
    "hwu": "HWU",
    "ecdt": "ECDT",
    "mcid": "M-CID",
    "DBPedia": "DBPedia",
}

# ========== 其它过滤 ==========
# plm_ood 目录别名
ALIASES_PLM = {"plmood", "llmood"}

# 只保留 args.fold_type == "fold"
FOLD_TYPE_KEEP = "fold"


In [18]:
import pandas as pd
import numpy as np
import json
import ast

# ------------------ 工具函数 ------------------
def parse_args_field(s: str) -> dict:
    """尽量鲁棒地把CSV里的args字符串解析成dict。"""
    if pd.isna(s):
        return {}
    txt = str(s)

    # 先尝试 json
    try:
        return json.loads(txt)
    except Exception:
        pass

    # 常见转义：{""k"":""v""}
    try:
        fixed = txt.replace('""', '"')
        if len(fixed) >= 2 and fixed[0] == '"' and fixed[-1] == '"':
            fixed = fixed[1:-1]
        return json.loads(fixed)
    except Exception:
        pass

    # 兜底：ast
    try:
        return ast.literal_eval(txt)
    except Exception:
        return {}

def _normalize_key(s: str) -> str:
    return str(s).strip().lower().replace("_", "").replace("-", "")

def rename_method(raw_name: str, fallback_task: str) -> str:
    key = _normalize_key(raw_name if pd.notnull(raw_name) else fallback_task)
    return METHOD_RENAME.get(key, str(raw_name if pd.notnull(raw_name) else fallback_task))

def method_display_name(m: str) -> str:
    return METHOD_DISPLAY.get(m, m)

def latex_escape_text(x) -> str:
    s = str(x)
    rep = {
        "\\": r"\textbackslash{}",
        "&":  r"\&",
        "%":  r"\%",
        "$":  r"\$",
        "#":  r"\#",
        "_":  r"\_",
        "{":  r"\{",
        "}":  r"\}",
        "~":  r"\textasciitilde{}",
        "^":  r"\textasciicircum{}",
    }
    return "".join(rep.get(ch, ch) for ch in s)

def fmt_kcr(x: float) -> str:
    # 0.50 -> 0.5，0.25 -> 0.25
    try:
        return f"{float(x):g}"
    except Exception:
        return str(x)

# ------------------ 读取与合并（含 plm_ood→Detector；fold_type过滤；坏CSV跳过并打印） ------------------
base = ROOT / SETTING
metrics = METRICS_MAP[SETTING]
KEEP_LCR_VALUE = LCR_KEEP[SETTING]

task_list = TASKS_MAP.get(SETTING, [])
if (not task_list) or (task_list == ["*"]):
    task_list = sorted(p.name for p in base.iterdir() if p.is_dir() and (p / "results.csv").exists())

frames, missing = [], []
print(f"[SCAN] base = {base.resolve()}")
for task in task_list:
    csv_path = base / task / "results.csv"
    print(f"[READ] {csv_path.resolve()}")

    try:
        # 用 python engine 更宽容；遇到坏行就跳过
        tdf = pd.read_csv(csv_path, engine="python", on_bad_lines="skip")
    except Exception as e:
        print(f"[WARN] 读取失败，已跳过：{csv_path} | {type(e).__name__}: {e}")
        missing.append(str(csv_path))
        continue

    # 解析 args
    if "args" not in tdf.columns:
        print(f"[WARN] 缺少 args 列，已跳过：{csv_path}")
        continue

    tdf["__args_obj__"] = tdf["args"].apply(parse_args_field)

    # fold_type 过滤
    tdf["__fold_type__"] = tdf["__args_obj__"].apply(
        lambda d: d.get("fold_type") or d.get("foldType") or d.get("fold-type")
    )
    tdf = tdf[tdf["__fold_type__"].astype(str).str.lower() == str(FOLD_TYPE_KEEP).lower()].copy()
    if tdf.empty:
        print(f"[SKIP] fold_type 过滤后为空：{csv_path}")
        continue

    # 统一 Method：普通方法重命名；plm_ood 用 Detector/Detecor
    if "method" in tdf.columns:
        def _method_from_row(row):
            raw_m = row.get("method", None)
            renamed = rename_method(raw_m, task)
            if any(_normalize_key(x) in ALIASES_PLM for x in (task, raw_m, renamed)):
                det = row["__args_obj__"].get("Detector") or row["__args_obj__"].get("Detecor")
                return det if det else "LLM-OOD"
            return renamed
        tdf["Method"] = tdf.apply(_method_from_row, axis=1)
    else:
        if _normalize_key(task) in ALIASES_PLM:
            tdf["Method"] = tdf["__args_obj__"].apply(lambda d: d.get("Detector") or d.get("Detecor") or "LLM-OOD")
        else:
            tdf["Method"] = rename_method(task, task)

    # 基本字段
    tdf["dataset"] = tdf["dataset"].astype(str)
    tdf["KCR"] = tdf.get("known_cls_ratio", np.nan)
    tdf["LCR"] = tdf.get("labeled_ratio", np.nan)

    # 清理临时列
    tdf.drop(columns=["__args_obj__", "__fold_type__"], inplace=True, errors="ignore")

    frames.append(tdf)

if not frames:
    raise FileNotFoundError("未找到任何可用 results.csv（或均被跳过）。\n失败文件：\n" + "\n".join(missing))

df = pd.concat(frames, ignore_index=True)
print(f"[INFO] 合并后总行数：{len(df)}")
print(f"[INFO] 合并后 Method 去重：{sorted(df['Method'].astype(str).unique().tolist())}")

# ------------------ Method 白名单（METHOD_ORDER_MAP 没写的直接不显示） ------------------
ALLOWED = [m for m in METHOD_ORDER_MAP.get(SETTING, []) if m]
if ALLOWED:
    df["Method"] = df["Method"].astype(str)
    df = df[df["Method"].isin(set(ALLOWED))].copy()
    if df.empty:
        raise ValueError(
            f"{SETTING}: 方法白名单过滤后无数据。请检查 METHOD_ORDER_MAP['{SETTING}'] 是否与实际 Method 名一致。"
        )
    print(f"[DEBUG] 白名单过滤后 Methods: {sorted(df['Method'].unique().tolist())}")
    print("[DEBUG] 白名单过滤后各方法行数：")
    print(df["Method"].value_counts().sort_index())

# ------------------ 过滤 LCR + KCR ------------------
df = df[df["LCR"] == KEEP_LCR_VALUE].copy()
df = df[df["KCR"].isin(KCR_ORDER)].copy()
if df.empty:
    raise ValueError(f"{SETTING}: 过滤 LCR={KEEP_LCR_VALUE}, KCR in {KCR_ORDER} 后无数据。")

# ------------------ openset 指标：<1 视为分数×100（只对要导出的指标列） ------------------
if SETTING == "openset":
    for m in metrics:
        if m in df.columns:
            df[m] = df[m].apply(lambda v: (v * 100.0) if (pd.notnull(v) and isinstance(v, (int, float)) and v < 1.0) else v)

# ------------------ 数据集白名单（11列） ------------------
wl = [d.lower() for d in DATASETS_ORDER]
df = df[df["dataset"].str.lower().isin(wl)].copy()
if df.empty:
    raise ValueError("数据集白名单筛选后无数据。请检查 DATASETS_ORDER 与 results.csv 的 dataset 命名。")

# ------------------ 聚合：对同一 (dataset, Method, KCR, LCR) 均值（自动平均 seed / fold_idx 等） ------------------
required_cols = ["dataset", "Method", "KCR", "LCR"] + metrics
miss = [c for c in required_cols if c not in df.columns]
if miss:
    raise KeyError(f"缺失列：{miss}")

grouped = (
    df[required_cols]
    .groupby(["dataset", "Method", "KCR", "LCR"], as_index=False)
    .mean(numeric_only=True)
)

# ------------------ 长表 + pivot：index=(Metric,KCR,Method), columns=dataset ------------------
long = grouped.melt(
    id_vars=["dataset", "Method", "KCR", "LCR"],
    value_vars=metrics,
    var_name="Metric",
    value_name="value"
)

pivot = long.pivot_table(
    index=["Metric", "KCR", "Method"],
    columns="dataset",
    values="value",
    aggfunc="mean"
)

# 列按 DATASETS_ORDER 排序（只保留存在的）
# 注意 dataset 大小写：用 lower 对齐后再映射回原列名
col_map = {str(c).lower(): c for c in pivot.columns}
datasets_cols = [col_map[d.lower()] for d in DATASETS_ORDER if d.lower() in col_map]
pivot = pivot.reindex(columns=datasets_cols)

# 增加 Avg 列（对 11 数据集求行均值）
pivot["__AVG__"] = pivot.mean(axis=1, numeric_only=True)

# 四舍五入
pivot = pivot.round(ROUND_DECIMALS)

# ------------------ 行排序：Metric 顺序；KCR 顺序；Method 按白名单顺序 ------------------
idx = pivot.index
metric_cat = pd.Categorical(idx.get_level_values(0), categories=metrics, ordered=True)
kcr_cat = pd.Categorical(idx.get_level_values(1), categories=KCR_ORDER, ordered=True)

method_whitelist_order = METHOD_ORDER_MAP.get(SETTING, []) or []
methods_present = list(idx.get_level_values(2).unique())
method_order = [m for m in method_whitelist_order if m in methods_present]  # 不追加其它
method_cat = pd.Categorical(idx.get_level_values(2), categories=method_order, ordered=True)

new_index = pd.MultiIndex.from_arrays([metric_cat, kcr_cat, method_cat], names=idx.names)
pivot = pivot.reindex(new_index).sort_index(level=[0, 1, 2])

# ------------------ 高亮：每个 dataset/Avg 列内，按 (Metric,KCR) 对 Method 排名 ------------------
def highlight_one_col(col_ser: pd.Series) -> pd.Series:
    out = col_ser.astype(object)
    num = pd.to_numeric(col_ser, errors="coerce")

    for (metric_val, kcr_val), sub in num.groupby(level=[0, 1]):
        order = sub.sort_values(ascending=False, kind="mergesort")
        if len(order) >= 1 and pd.notnull(order.iloc[0]):
            out.loc[(metric_val, kcr_val, order.index.get_level_values(2)[0])] = (
                "\\textbf{" + f"{order.iloc[0]:.{ROUND_DECIMALS}f}" + "}"
            )
        if HIGHLIGHT_TOP2 and len(order) >= 2 and pd.notnull(order.iloc[1]):
            out.loc[(metric_val, kcr_val, order.index.get_level_values(2)[1])] = (
                "\\underline{" + f"{order.iloc[1]:.{ROUND_DECIMALS}f}" + "}"
            )

        for idxi, v in sub.items():
            if pd.isnull(v):
                out.loc[idxi] = ""
            elif out.loc[idxi] == col_ser.loc[idxi]:
                out.loc[idxi] = f"{v:.{ROUND_DECIMALS}f}"
    return out

formatted = pd.DataFrame(index=pivot.index)

# dataset列 + Avg列
all_cols = list(pivot.columns)
for c in all_cols:
    formatted[c] = highlight_one_col(pivot[c])

# ------------------ 生成 LaTeX（tabularx + multirow + cmidrule + Avg） ------------------
# 表头显示名
dataset_headers = [DATASET_DISPLAY.get(d, str(d)) for d in DATASETS_ORDER]
# 真实列名按 datasets_cols 对应顺序；用于取数
dataset_real_cols = datasets_cols

# 总列数：Metric(1) + KCR(2) + Method(3) + 11 datasets + Avg = 15
total_cols = 3 + len(dataset_headers) + 1
cmid_l = 2
cmid_r = total_cols

# tabularx 列格式：@{} l l l | l *{10}{Y} | Y @{}
# 第一个dataset用 l，其余用 Y，最后 Avg 用 Y，并在 Avg 前加竖线
n_y = max(0, len(dataset_headers) - 1)
colspec = f"@{{}}l l l |l *{{{n_y}}}{{Y}}|Y@{{}}"

# caption / label（caption 用 XXX 占位）
label = f"table::{SETTING}_main"

lines = []
lines.append("\\begin{table*}[ht!]")
lines.append("\\caption{XXX}")
lines.append(f"\\label{{{label}}}")
lines.append("\\centering")
lines.append("\\tiny")
lines.append("\\setlength{\\tabcolsep}{4pt}")
lines.append("")
lines.append(f"\\begin{{tabularx}}{{\\textwidth}}{{{colspec}}}")
lines.append("\\toprule")

# header 行
header_cells = ["Metric", "KCR", "Method"] + dataset_headers + ["\\textbf{Avg.}"]
lines.append(" & ".join(header_cells) + "  \\\\ \\midrule")

# 逐块写：Metric -> KCR -> Method（multirow）
metrics_in_table = [m for m in metrics if m in pivot.index.get_level_values(0).unique()]
kcr_in_table = [k for k in KCR_ORDER if k in pivot.index.get_level_values(1).unique()]

for mi, metric in enumerate(metrics_in_table):
    # 本 metric 下每个 kcr 的 methods
    blocks = []
    for kcr in kcr_in_table:
        methods_block = [m for m in method_order if (metric, kcr, m) in formatted.index]
        if methods_block:
            blocks.append((kcr, methods_block))

    if not blocks:
        continue

    metric_span = sum(len(ms) for _, ms in blocks)
    metric_written = False

    for bi, (kcr, methods_block) in enumerate(blocks):
        kcr_span = len(methods_block)
        kcr_written = False

        for mj, method in enumerate(methods_block):
            row_idx = (metric, kcr, method)
            row = formatted.loc[row_idx]

            # Metric cell
            if not metric_written:
                metric_cell = f"\\multirow{{{metric_span}}}{{*}}{{{latex_escape_text(metric)}}}"
                metric_written = True
            else:
                metric_cell = ""

            # KCR cell
            if not kcr_written:
                kcr_cell = f"\\multirow{{{kcr_span}}}{{*}}{{{fmt_kcr(kcr)}}}"
                kcr_written = True
            else:
                kcr_cell = ""

            # Method cell（显示名 + 转义）
            m_disp = latex_escape_text(method_display_name(method))

            # dataset values
            vals = []
            for real_c in dataset_real_cols:
                v = row.get(real_c, "")
                vals.append(v if isinstance(v, str) else ("" if pd.isnull(v) else f"{v:.{ROUND_DECIMALS}f}"))

            # Avg
            vavg = row.get("__AVG__", "")
            avg_cell = vavg if isinstance(vavg, str) else ("" if pd.isnull(vavg) else f"{vavg:.{ROUND_DECIMALS}f}")

            lines.append(" & ".join([metric_cell, kcr_cell, m_disp] + vals + [avg_cell]) + " \\\\")

        # kcr block 分割线（和你示例一致用 cmidrule）
        if bi < len(blocks) - 1:
            lines.append(f"\\cmidrule(l){{{cmid_l}-{cmid_r}}}")

    # metric block 分割线（最后一个 metric 不加 midrule）
    if mi < len(metrics_in_table) - 1:
        lines.append("\\midrule")

lines.append("\\bottomrule")
lines.append("\\end{tabularx}")
lines.append("\\end{table*}")

latex_table = "\n".join(lines)
Path(OUTPUT_TEX).write_text(latex_table, encoding="utf-8")

print(f"[OK] V5 主实验表已生成：{Path(OUTPUT_TEX).resolve()}")


[SCAN] base = /ssd/lijinpeng/code/bolt/results/openset
[READ] /ssd/lijinpeng/code/bolt/results/openset/ab/results.csv
[READ] /ssd/lijinpeng/code/bolt/results/openset/adb/results.csv
[READ] /ssd/lijinpeng/code/bolt/results/openset/deepunk/results.csv
[READ] /ssd/lijinpeng/code/bolt/results/openset/doc/results.csv
[READ] /ssd/lijinpeng/code/bolt/results/openset/dyen/results.csv
[READ] /ssd/lijinpeng/code/bolt/results/openset/knncon/results.csv
[READ] /ssd/lijinpeng/code/bolt/results/openset/plm_ood/results.csv
[READ] /ssd/lijinpeng/code/bolt/results/openset/scl/results.csv
[INFO] 合并后总行数：13856
[INFO] 合并后 Method 去重：['AB', 'ADB', 'DOC', 'DeepUNK', 'DyEn', 'EnergyBased', 'Entropy', 'KLMatching', 'KNNCon', 'LogitNorm', 'Mahalanobis', 'MaxLogit', 'MaxSoftmax', 'OpenMax', 'SCL', 'TemperatureScaling']
[DEBUG] 白名单过滤后 Methods: ['AB', 'ADB', 'DOC', 'DeepUNK', 'DyEn', 'EnergyBased', 'KNNCon', 'MaxLogit', 'OpenMax', 'SCL']
[DEBUG] 白名单过滤后各方法行数：
Method
AB              791
ADB             582
DOC       

/tmp/ipykernel_1057465/2045070495.py:227: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for (metric_val, kcr_val), sub in num.groupby(level=[0, 1]):
/tmp/ipykernel_1057465/2045070495.py:227: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for (metric_val, kcr_val), sub in num.groupby(level=[0, 1]):
/tmp/ipykernel_1057465/2045070495.py:227: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for (metric_val, kcr_val), sub in num.